In [1]:
import numpy as np
import pandas as pd 
from sklearn.metrics import *
from sklearn.model_selection import *
from lightgbm import LGBMClassifier
import gc

In [3]:
print("Input Data Files :\n")
!ls input_data
print("\n\nOutput Data Files :\n")
!ls output_data

Input Data Files :

SampleSubmission.csv  test_customers.csv  train_customers.csv  vendors.csv
orders.csv	      test_locations.csv  train_locations.csv


Output Data Files :

__notebook__.ipynb  __results__.html  test.pkl
__output__.json     custom.css	      train.pkl


In [4]:
INPUT_PATH = './input_data/'
OUTPUT_PATH = './output_data/'
sample_sub = pd.read_csv(INPUT_PATH + 'SampleSubmission.csv')
train = pd.read_pickle(OUTPUT_PATH + 'train.pkl')
test = pd.read_pickle(OUTPUT_PATH + 'test.pkl')

In [5]:
train.shape, test.shape

((5950300, 63), (1672000, 63))

In [6]:
train['target'].value_counts(normalize=True)

0    0.986531
1    0.013469
Name: target, dtype: float64

In [7]:
gc.collect()
drop_cols = train.head(100).select_dtypes('object').columns.tolist()
len(drop_cols)
ID_COL, TARGET_COL = 'CID X LOC_NUM X VENDOR', 'target'
features = [c for c in train.columns if c not in [ID_COL, 'target', 'customer_id'] + drop_cols]
trn, val = train_test_split(train, test_size= 0.2, stratify = train[TARGET_COL], random_state=1)

In [8]:
del trn
gc.collect()

20

In [9]:
### Intially model was only trained with 80% data to get the optimal threshold.
### Next the model was trained with complete data
clf = LGBMClassifier(n_estimators=2000, max_depth = -1, learning_rate=0.07, colsample_by_tree = 0.8, reg_alpha=1, reg_lambda=1, random_state=3)
clf.fit(train[features], train['target'], eval_set = [(val[features], val['target'])], verbose=50, early_stopping_rounds=150)

Training until validation scores don't improve for 150 rounds
[50]	valid_0's binary_logloss: 0.0570814
[100]	valid_0's binary_logloss: 0.0547988
[150]	valid_0's binary_logloss: 0.0537824
[200]	valid_0's binary_logloss: 0.0530741
[250]	valid_0's binary_logloss: 0.0524061
[300]	valid_0's binary_logloss: 0.0520312
[350]	valid_0's binary_logloss: 0.0517583
[400]	valid_0's binary_logloss: 0.0515297
[450]	valid_0's binary_logloss: 0.0512908
[500]	valid_0's binary_logloss: 0.0510653
[550]	valid_0's binary_logloss: 0.0508946
[600]	valid_0's binary_logloss: 0.0507147
[650]	valid_0's binary_logloss: 0.0505596
[700]	valid_0's binary_logloss: 0.0504087
[750]	valid_0's binary_logloss: 0.050271
[800]	valid_0's binary_logloss: 0.050144
[850]	valid_0's binary_logloss: 0.0500338
[900]	valid_0's binary_logloss: 0.0499098
[950]	valid_0's binary_logloss: 0.0497947
[1000]	valid_0's binary_logloss: 0.0496826
[1050]	valid_0's binary_logloss: 0.0495799
[1100]	valid_0's binary_logloss: 0.0494804
[1150]	valid_0

LGBMClassifier(boosting_type='gbdt', class_weight=None, colsample_by_tree=0.8,
               colsample_bytree=1.0, importance_type='split',
               learning_rate=0.07, max_depth=-1, min_child_samples=20,
               min_child_weight=0.001, min_split_gain=0.0, n_estimators=2000,
               n_jobs=-1, num_leaves=31, objective=None, random_state=3,
               reg_alpha=1, reg_lambda=1, silent=True, subsample=1.0,
               subsample_for_bin=200000, subsample_freq=0)

In [10]:
#### Lets see some thresholds
val_preds = clf.predict_proba(val[features])[:, 1]
for i in range(1, 50, 1):
    thresh = (i / 100)
    print(str(thresh) + ': ' + str(f1_score(val['target'], (val_preds > thresh)*1)))

0.01: 0.09022281891840965
0.02: 0.12658365062804028
0.03: 0.16024198980694196
0.04: 0.18908667443021127
0.05: 0.21241030320627152
0.06: 0.23287877646349736
0.07: 0.24946989130918315
0.08: 0.26339591612991636
0.09: 0.27362575483971185
0.1: 0.2810454683368731
0.11: 0.28518793214824034
0.12: 0.288282168517309
0.13: 0.2892169411497
0.14: 0.28508422664624805
0.15: 0.28179053722656383
0.16: 0.2770679685606455
0.17: 0.2699042557064327
0.18: 0.26337668281148696
0.19: 0.2546246781540899
0.2: 0.24677882642593355
0.21: 0.23883626522327472
0.22: 0.23067569914886227
0.23: 0.2211726964890394
0.24: 0.21472896513962397
0.25: 0.20655523391539826
0.26: 0.19765110283586365
0.27: 0.18736913862784244
0.28: 0.1770657672849916
0.29: 0.16775170325510974
0.3: 0.15953626756950856
0.31: 0.15197790170427894
0.32: 0.14281183932346722
0.33: 0.13502652306703103
0.34: 0.12718773779758666
0.35: 0.11828134455630741
0.36: 0.11142650414649079
0.37: 0.10455236058747397
0.38: 0.09854118181302153
0.39: 0.0930073126142596
0.

In [11]:
test_preds = clf.predict_proba(test[features])[:, 1]

In [12]:
thresh = 0.12 ### OPTIMAL THRESHOLD
pd.Series((test_preds > thresh) * 1).value_counts()

0    1641393
1      30607
dtype: int64

In [13]:
icol = 'CID X LOC_NUM X VENDOR'

In [14]:
sub_df = pd.DataFrame()
sub_df[ID_COL] = test[ID_COL]
sub_df['target'] = ((test_preds > thresh) * 1).astype(np.int16)
sub_df = pd.merge(sample_sub[['CID X LOC_NUM X VENDOR']], sub_df, on = 'CID X LOC_NUM X VENDOR', how = 'inner')
sub_df['target'].value_counts(normalize=True)
SUB_FILE_NAME = OUTPUT_PATH + 'zindi_weekend_food_recommendation_final_solution.csv'
sub_df.to_csv(SUB_FILE_NAME, index=False)

In [15]:
print("Input Data Files :\n")
!ls input_data
print("\n\nOutput Data Files :\n")
!ls output_data

Input Data Files :

SampleSubmission.csv  test_customers.csv  train_customers.csv  vendors.csv
orders.csv	      test_locations.csv  train_locations.csv


Output Data Files :

__notebook__.ipynb  test.pkl
__output__.json     train.pkl
__results__.html    zindi_weekend_food_recommendation_final_solution.csv
custom.css


In [16]:
from IPython.display import HTML
def create_download_link(title = "Download CSV file", filename = "data.csv"):  
    html = '<a href={filename}>{title}</a>'
    html = html.format(title=title,filename=filename)
    return HTML(html)
create_download_link(filename = SUB_FILE_NAME)